# MLflow Autologging: Autologging artifacts and plots

This notebook is divided into small cells with markdown explanation and line-by-line comments. It uses `datasets/customer_churn_500.csv` from this folder.

## 1. Import required libraries

This cell imports MLflow, pandas, NumPy, matplotlib, and scikit-learn utilities.

In [ ]:
# Import os to create local folders and manage paths.
import os

# Import warnings to hide non-critical warning messages during training.
import warnings

# Ignore warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking and model management.
import mlflow

# Import MLflow sklearn integration for logging sklearn models.
import mlflow.sklearn

# Import pandas for reading and processing CSV data.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for creating charts and artifact images.
import matplotlib.pyplot as plt

# Import train_test_split for splitting data into train and test sets.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for separate numerical and categorical processing.
from sklearn.compose import ColumnTransformer

# Import encoders and scalers for preprocessing.
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to combine preprocessing and model into one object.
from sklearn.pipeline import Pipeline

# Import ML models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import regression metrics.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Configure MLflow

This cell configures MLflow to store runs locally in the current folder under `mlruns`.

In [ ]:
# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Store tracking data locally in this folder.
mlflow.set_tracking_uri("file:./mlruns")

# Set experiment name for this notebook.
mlflow.set_experiment("07_mlflow_autologging_05_autologging_artifacts_and_plots")

# Define the dataset path used by this notebook.
DATA_PATH = "datasets/customer_churn_500.csv"

# Print confirmation.
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)

## 3. Load and inspect dataset

The dataset contains 500 realtime-style customer records with categorical and numerical columns.

In [ ]:
# Read the CSV dataset from the local datasets folder.
df = pd.read_csv(DATA_PATH)

# Display the first five records.
display(df.head())

# Display dataset shape.
print("Dataset shape:", df.shape)

# Display column names.
print("Columns:", df.columns.tolist())

## 4. Prepare features and target

For classification, we predict `churn`. For regression, we predict `monthly_charges`.

In [ ]:
# Define classification target column.
target_column = "churn"

# Remove customer ID and target column from features.
X = df.drop(columns=["customer_id", target_column])

# Store the classification target.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature summary.
print("Target column:", target_column)
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

## 5. Build preprocessing pipeline

Categorical columns are one-hot encoded. Numerical columns are scaled.

In [ ]:
# Create preprocessing logic for numerical and categorical columns.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # Convert categorical columns into numeric one-hot encoded columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Print the preprocessing object.
preprocessor

## 6. Split the dataset

Training data is used to train the model. Testing data is used to evaluate the model.

In [ ]:
# Split data into training and testing datasets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print split sizes.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Enable autologging

Autologging captures model parameters, metrics, and artifacts automatically.

In [ ]:
# Enable MLflow autologging for scikit-learn.
mlflow.sklearn.autolog()

# Print confirmation.
print("MLflow sklearn autologging is enabled.")

## 7. Create and train model

This cell builds a complete scikit-learn pipeline and trains it.

In [ ]:
# Create classification model.
model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)

# Create complete ML pipeline.
pipeline = Pipeline(steps=[
    # First apply preprocessing.
    ("preprocessor", preprocessor),

    # Then apply model.
    ("model", model),
])

# Train pipeline on training data.
pipeline.fit(X_train, y_train)

# Generate predictions on test data.
predictions = pipeline.predict(X_test)

# Print sample predictions.
print("Sample predictions:", predictions[:10])

## 8. Evaluate and log with MLflow

This cell creates an MLflow run and logs parameters, metrics, artifacts, and model.

In [ ]:
# Start an MLflow run.
with mlflow.start_run(run_name="classification_training_run"):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log business problem.
    mlflow.log_param("business_problem", "customer_churn_prediction")

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Calculate accuracy.
    accuracy = accuracy_score(y_test, predictions)

    # Calculate precision.
    precision = precision_score(y_test, predictions, zero_division=0)

    # Calculate recall.
    recall = recall_score(y_test, predictions, zero_division=0)

    # Calculate F1-score.
    f1 = f1_score(y_test, predictions, zero_division=0)

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Create classification report.
    report = classification_report(y_test, predictions)

    # Save classification report locally.
    with open("classification_report.txt", "w") as file:
        file.write(report)

    # Log classification report artifact.
    mlflow.log_artifact("classification_report.txt")

    # Log trained model.
    mlflow.sklearn.log_model(pipeline, "model")

    # Store run and model URI.
    run_id = mlflow.active_run().info.run_id
    model_uri = f"runs:/{run_id}/model"

# Print logged details.
print("Run ID:", run_id)
print("Model URI:", model_uri)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

## 9. Create and log visual artifact

This cell logs a churn distribution chart as an MLflow artifact.

In [ ]:
# Create churn distribution chart.
plt.figure(figsize=(6, 4))

# Plot target class counts.
df["churn"].value_counts().sort_index().plot(kind="bar")

# Add chart title.
plt.title("Customer Churn Distribution")

# Add x-axis label.
plt.xlabel("Churn Class")

# Add y-axis label.
plt.ylabel("Number of Customers")

# Improve chart layout.
plt.tight_layout()

# Save chart as PNG.
plt.savefig("churn_distribution.png")

# Close chart.
plt.close()

# Start another MLflow run for artifact demo.
with mlflow.start_run(run_name="visual_artifact_run"):

    # Log the chart artifact.
    mlflow.log_artifact("churn_distribution.png")

# Print completion message.
print("Visual artifact logged: churn_distribution.png")

## Final trainer note

Ask students to open MLflow UI from this folder using:

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open `http://127.0.0.1:5000`.